# P1 — E1: Does expensive self-calibration improve a fixed scheduling attack?

**Independent, resumable Qoala/NetSquid evaluation.** Uses the original frozen Q12 source and calibration records **read-only**. Does not need `P1_Resume_After_MemoryError`. Start with `MODE="pilot"`; inspect its RAM and wall-time report before choosing `MODE="full"`.

## Important correction
Q12 already learns **one fixed phase** from 16 candidates. A separate “calibration-selected fixed strategy” with the same budget would be the **same policy**, not a new comparator. E1 therefore measures the *value of per-campaign self-calibration* against cheaper or shared fixed schedules.

| Policy | Calibration used to select its fixed phase |
|---|---|
| `q12_full256` | Frozen Q12 choice: 256 own executions for each of 16 candidate phases |
| `low08`, `low32`, `low64` | First 8, 32, or 64 original *own* observations per phase; no new calibration |
| `global_loo` | Pooled own observations from the **other 31** calibration campaigns; fixed phase transferable across campaigns, substantial amortized training cost |
| `fixed_0ms`, `fixed_2ms` | Simple predetermined phases; 2 ms was examined in earlier exploratory work and is **not** a new preregistered discovery |
| `random_phase` | Equal-work random release phase |
| `no_competitor` | No competing work; reference only, not an equal-work timing baseline |

**New evaluation:** teleportation only, 32 campaigns, 256 fresh shots per arm per campaign, ±0.5 ms supplied request-epoch uncertainty, same FCFS processor and four competing jobs totaling 1.2 ms. Nine policies × 32 campaigns × 256 = **73,728** fresh victim executions plus **1,152** integration-pilot executions. Original Q12 evaluation RNG is never reused. The two preselected primary contrasts are full Q12 versus `global_loo` and versus `fixed_2ms`; other comparisons are descriptive/exploratory. Inference uses campaigns, not shots, as its replicate units.

**Runtime and safety:** with eight 128-case worker batches in flight, Q12's previous peak was approximately **1.41 GiB per worker**. Reserve **at least 15 GiB actually available** and preferably a 24–32 GiB machine when E1 is the only memory-heavy task. The current memory-error recovery job may use this headroom; if so use fewer workers or a separate server. The pilot measures actual resource use before the full run. Do not modify Q12–Q16 or their results.

**Validation:** Local source/policy tests have passed; the new E1 integration has **not** been executed with NetSquid in this environment. Only the server pilot can establish simulator compatibility. Do not interpret pilot success as experimental support for the attack.

In [ ]:
from pathlib import Path

# Existing Q12 artifacts on your server. Neither is modified.
PROJECT_DIR = Path.home() / "P1"
Q12_RUN_DIR = PROJECT_DIR / "results" / "q12-20261013T034634906569Z"
Q12_CODE_DIR = None  # Normally auto-located from Q12's pinned source hash.

# Start with pilot. After it passes and sufficient resources are free, change to "full".
MODE = "pilot"  # "pilot" or "full"
WORKERS = 8    # 8 processes require 8 available CPU cores and substantial FREE RAM.
RESUME_RUN_DIR = "auto"  # "auto" reuses this E1 run; None starts a NEW E1 run.

# Set only if the existing pinned Q12 environment is in a nonstandard location.
PYTHON = None
REPO = None

# Review export: compact evidence and metadata; full raw traces stay on server.
EXPORT_COMPACT = True


## Install the pinned E1 runner (no change to Q12)

In [ ]:
import base64, gzip, hashlib, json, os, sys
from pathlib import Path

E1_BUNDLE_SHA256 = "b1a9477d54794bab92ed32a30e8f0736dd4ee791bc49e9cf817e960a87426034"
_E1_EMBEDDED = "H4sIAAAAAAAC/609DXPbNrJ/hZebDsmEoiX5I6kS3IyTOq3nmthnu3fvRtZwKBGyWFMkS1L+qM/vt7/dBUCCFKi4vde7iUUQWACLxX4DfHqVjwI+8vPHV5NX1/S/85F1MppY67C4idMwse7CZMOtbGnlvBgswnUexjeptYwfeDTIV2HJrZInS3iTxPMirOIs9a/T6/SCb0peWtWKW3mcpjyy/jEaWxyhhVVWWPdxtco2lbVYhSl0dGPFVWmV2aZY8NKzynhtLbJ0Gd9sBEzvOoVGWRGLQWm97UmY8NMqeLlJqtK3rqDbMLrjRRkWjxaNJEuTR+oku4fhaQCsbF7y4o5+Q9OvJ/+y7uJFBSNY87DcFHzNU2gGQKzQiuKyitNFBZOGKRVZVsnZ/raJCznfepQ442KTQqOCL2DSj1aYRlQlTssqTBIeedYmXWdRvIwlhhZZxAHk18zKwwI6BkwnABd+W/dFXFU89a0LMU0qjNfrTRXOE06wEQFrfMJRiQVNl0W2toJgualgLkEALfKsqKB6mlVi0tepKituoNeS1wWwBotNgePwRfum7nKRVkn9dPN7nNcPQBUrQG/9/GuZpfVDVsoh5WGFtdR4zuGxrgQdESnUBeVqU8VNdyXQYKg9buZ5kQHlNMMrH5vf1argYQRE1pTEay6HschgHRaEBzWUiC9DQHAEVICL+9PJxYnFaIAO4BHWIwhcH4eY3HHH9cVCXaeXV2fnUK/uzT+5g3LHvU7/eXJxeXr2FV7a+WjAR/puGSyyshrcjezr9Pz44uTrVfDT8eVPWPXg7XD49l04P5iPF8PRQXQwnn//NprPw4Pvj77fPwiHo+8PQz7aDw/fRfsHw4Px24PF8Gj+jh/O9+dv5wcAESgqOD+ljvdHy3C8gJbfH759F43Hb9/uHx0evl3uHy6i4Sgc7R8sDubziEOrT8dfzo9Pf/x6Ce32xzCxn86u8Pf48Og6/Xh89QmHNxq/gxGf/nx2Fej160LVaHSkSr6enV6eBJ8vTk7ql1Ab9lsg3l+cnV1hL2+Ho/HRcCRenfzz+OfOGxjRRyi4vLo4Pu+82r9Ojy++IGTnOrXgP/s3wMBykyQwdtuz7CS7H76TP/bH8sfRAf64SbJ5mARJltmebExcLhiuS3wvHsbioYDdlq0DYoD4nGbBIlvnvIphnwMKYdHPYR1PgvOzn08/nZ7gkEpeOTg6d/DUad9p/gyNL06/HF/8O/h0Bqi9OL46u6BJtQapjchFOr1OgXCtX6PNOney+a/uRMyi4LBxU9qEPr4r8SWwWKD04JY/luyq2HB45kDHyJpL5gBsmNPEdj0LuFR2H6Rhyj6HScm1jqL4hpeVoSe5/f1yFQLWnWZAPk+RvTmu66/4g2yvQcSthY0dZA0K6Arm3YHoijcoQsSupPp+lvPUsYu57VphaS1XEgD+twThMd8sge+CCOCFk4TreRROlisf96ozGo4PXuM/MOG5bbtaSxqCv8mjsOIOgHDbUzVPhIDqk9DXIMnCqHS0cWPtoOIPAEKDUTN2quVZGp6xQDEkAtEUS2bkr29B6DjiQa0wfwDhFWS39CjbxEvRjN6Vjj7zWJ+H9ReGI+ggpghjkIoXmxT56UlRZIVjnzyQiLzRJNOaVyHgD6RnBqIMxA5oF9ViNbHsN2VViA7cBrJAlniu1jnMlIaIyw2EuBYI8fHXG9uHCgMBJiv9G17lcQRo1CkEakjSuDdQRr0taFcsV14M2E8rNvY6G2R7I9TEBWMD0cwd+/o6tdsvlskG6Fkrg1Euy8d04eBLoPc0c9Ro4VXB8yRccAfG7MmVVfQgBGJAy1pkv4LI8ggvQQZqThFHnH3NUt5HInUtVy15XWJxmI6sKQC7ezZ/AHUvRs0nwC1r7znISQHTmoyaTkZHM7fVn6SlHLgb6EuFo4nJmt5w/aleXNJsdJozEdSp0pVIPxJosNZxWSKRhZVOREAP2FhUBCrKgaUUZeWrRQGlwBddpzC6yhl6Df3h0OonGiQwC9VA4AgRLKchFAUULhFHZQTpKuq+w0KlDIp3YYn9WpFPQplZSj6j7hb5tZrASG4NULEGTWGAugG2xpVAVgibiNWAfSwNRLHT5k6CcYCOqep6GoiGsgSXCGKk+3j5KLmGXrehKlJJa7rCJ3fnegNFVRmoWNBGMhNss2erch/3n1oeFIEJr2ClO7VRCv58cnXSrq2p8N36xz+ffgSpCcgMjG1B8/udbzUTpQGgXSqErTZy7dTIpzaaF1DLnjHWWS5az6Yi7OosACsIa8oV7wGprSVW1ldLq1/j6U80gL1UbaCuDz+KqkQOiTybLzaESFWzM2lNMu9GFQyhWRborX4fCNltz/403GYOW5VeCFwbmr1nr8M0XoLgNo9cewhUTXM3wMMdMSKAcXRAaw/8yrErGB+ygv8IDhW7yFEk5aFCEuMzKII33Kk1aLe9g9EEcZDLgBmJzJIJDkX7zpP7VlGPHB3bmnZnr3ltGd7ZSt3Jsj+GRQPwvtViL158A1DEiaKIXnidre9hIylGxSbR9ovrid4btpjdp7wQGnqguwhq/qgxVsUghc19AUzFAu7QWPUAa6DBsKSgJaMTHROelXJoYMGeBB3GalwavrLiSUHQRsE0I9XBf2oeCsKTDSX3pcEn4RzEF3uyxeN3w3Fkfxd3SXB05D6LVnNUzgA5qADxyHF0Zt9eftdHe8SxqcXg9V69RyXG3a2NImHDThnvDw/AyPglRVVjUSn3h8bUoeLiNs/itBKzklIQx70Oi1tcHBTNMAMJtaXwJxEvmFavNtNrzS+bM2L/ou6eDQUtho///dX6SguDFWmA0jXUrJAH+LtHh9UmIcdWVYTkwoKfYfpoZdUKmgOzvUGfjAJbwpRLBh0Ci8SfiqVIFQnZRzm177PiFq0Fe/YXVnMTmxBQknaCTbvGyiIDKZ5ueFMq0S+B0lBIZukrSVyr1SXb0eUWbJpJRxbtUlDwP7E2Ygm0ddoGLl5OcX0IDAKPUH6hcQSFWotFzuRqqkZIkuGiIreNjuWujFjkQJPdRm1+X1sU6OwSJsUi92zAz7ZZoQgVzTpi+92XgskvmGYSYl13uxbsekDSYmoDE9EZkT6shlzP0M2JLsawqsIFzMYupTCBHTaPK1roBfwTo0W7VxVxmFiCQ1jlpriL73Q67WALhwKo18WFNiBiQw5SiBgv/rJnYFDlmwrZj2BuAfFVe2KrAQY6HT4beH1LNWSomc+zLHGoj026jNMYSAso1vXuiyy9CbJNBT0y4Buijl4KtVwDjmHPwURQ6kvE2DNlAug8dNI3uO6W28abICqaAsDGRaBCtTQBLA2UQ39glIwMI1x4YpU8q1xlFZOjFd54QHJ39KIAq5rIRA5r+IEtPjTOOxwUFGGjD+Tq6za7Zc7Co1G4vTCxtUKdLq+mtzPPsaNNnsQLGKHV44C3vVsPQZj3gQ5titVmtMoGTJowiDLkDRt1NGMUl6xGwuvR0WuYOwxVH6CIcwAxZUVU2h41chtptDApch3vU1vSdghJ4Y7TXJpZLry2zJ4Bj8JKAg6Ms6Mu6o0bRWa+iZMoyDPAfMzLVg8eKVJC8wGFpq3GnGOLRxz0LZraQkMpOUY5umqKBbsL6BvDLMWWLGzpMCCFghLwCPrF07OmpZQsiUvQZHxym14GgME/huBb2jzvvP2xd3TgIW46SJa9TsvN+htYnpazRuKJbm9d17CMna11H6cpYIqt49Sp63i3/JEJf6MVT5yBGMY0BsbY5UQNbmBMtzPmCMxMJdyZJ3946/DBERVd1yTnGyiAh9l0CAKzWedp1zBZuLOpxEAKkhuUMlJZVRRLWixiD6zjknx3UhcjonqsV/IlawU0kZZLEPzT2R/YIaqVH+YgdyMHF3FrIUnX6i6mTwHM0jGx/S1TBWOMpLAZ5kBCAl/+hS10YHOwfnYvuRr81qIL9E3lIrCn1ghboYuJaVXbslLGNto135nr7Y879fbH5opHB52KRweGilpUYiJpFrHSrdaEUyZD4ysMZUzGQ/qvU6EVMJmgJ65ToR1C6dZ43rJEBO5djffXljuy2DvJailgQ4RxR9oAterSlKZh39U6Q0mtiuze0E5wLP7oUXWo48cVX+MLIDIox9J2/Kjj2SNwWrCgAj2qCsvb0iH9Xm0f8mnCMyjzeZyAMtBhmtpm7YTyXIMyGxZrrI8oMWhCnT2sRQHdHr3pMeZJJHwcSpVhoOdIpgK9SWUn9r65ddMsLjmz6Q/wOc5tnHv8oSfySO5cO+K4UMBQ7W93IAgUvWecxd8dfRs6eeMFWJ5o69AIzR2L8f+3DH9iATT8S/RL7Gr40tHRmmnJt9z5/0SiF858QZwN3SLJBovVJu0SLtGyUAm2aLu9Fab0ahpP4jcUmp51MTD0cLtTLdejKii31QhIl2sCX7rfjXYM6+yL1wiMmMJrjcIN9NOsMzM1Fs2wUqD8LWqYCiE6qbgmJxiskMQRwfTUGpZNj7SCZZADCqF2UK+yIAst9Cpcb8pV6LRiAtWjpAkTlqQXnMngBTrZWLe1pZRP5aqTIVMJ1TA1TK8JML0mEKvQzhLwmtcamlvpAgaYYDVWIGrDnFqydg6BJ7SGTRqDrWqnYZqVoO6nqO5vQxIxH2Z/FqoRqkujo55sKAv1dHQAWmRrl5hHJQSEiz7CclOCgh6BOj1I4nWMfjByxawAhZHgOlKcmAYCYjmHjXklFTrpkPr86fOlZ12N2YG1Lq1613rWm73B0D/EwnKTgyUGsKHdYmVt0gUIsBCMKVisZbYBZpSAUANTLUHfDmj8WQVTAitg5I+ttXEsYhaC1mjEYPK3VBi0q5SS8s5Tagjo60rRODowmf+NcgETBY6CGrAFeAVpK5W1/ZFVkz5hWkM/hr0XyQZWzFoBsxtQzpnKYpMCXzn1lBHz3grXgM34d8APmrK/8yITGEA9VAwAs3ZMWKgVGWafF/wuzjZl8ogeghJ2Q2dJ31tfz66sG55u4pRDrbyAGjewn0VFzE4qFwWp24ae8iJeh8UjaT0qc4OYgSF3xN3RHtRpcrIy4SiRjuuFnlGH+MBgK2J8uaTRcRUyAhuyIOJDrDTuGJVPAvMyDT4Ey+KxjBFLYYzw9sdNcmG9Ua1Pp++tj1kKXcLQYiuMft2U1ZrS4oDJj9UcyANThGWFS1eTRV0Ig82TjFBkHIyQAUyJAk/GCLphBcwD27NVwqTdStZQMepAmU2OAFJHThl50WWhISpOgZ2tuPOkJ2KtZODL4tCwxDAmju/Er28HqtW7o6DguODAH7DSb0fbcexOTFPJBk905dWxZ2yr8IVuZl44YXFT+69rdzwW+o1b1wSIGfBNLmgtwhPFRe0GUm44o0easbppy4NtjiV2iUC25qOmsWrK0zv225FPYYwA94iYGv5qj0vEAYRfnrwYUln3WhqARIeP0upyqr+adcQj0pMOs9b+pWaqy0mZGIHBYB4xTLvwYa+DFpDGC8d9r94Ei3wj3spEywAfFP2SzxfVwPeWTBdlTa5EVTxOeh3oAiPhvWc/VOhYWwMPLMuE3/GE7ZNbHV92G0j/fH8jWcGgPMcebTDguWuO7EnQDWmHQC4GPXnBKt3T+h61LiwSuovJuyril4BNAi2F90xZ+VNob2g0h1oIuW3oUjAISrvFwn5pm7xN3U65qJwA+JRHtmG8uHqRj39qxt9SPr0mEuShs0wKjuHhcLgBVUAO3auUw9k19zEV5hnQY1X/NLri4bVmX+CeEFljSFMTAan1nnUb9AWhMQjR9X1Nnhrf10Qt3fMz8WT5pPcvsKkcXi1CkZQv2JVfbFKKhci4sskhjYwByEQ2AHyT896RgLzdrZUDWwAR4QWx4gG6xOKF8OYx5gx3U0bkf/zl4hIMmZOLf55+Omk8r7t6U6gJGr8h+ktEYd+iquVv2ek9pmm3w3YQh7GhCGSp13ociKnwtwHZxKnV1kUTd3dN5AtIX4bYXXgvs/5EfquwGDXPBTZllXCwQ2UmV9V133TTBPUYFzCtFlg5mP5GgFknfjNyv9sfw7wnsDfTCjYvaWeADA/eefaeTTZwi9V5lJqo54KCoszzyvoIC3pCP2Hvt/NBkSJbm7FJUlWMXEvTrItkquYyBCEaDRCIysjA326rC4mm/7oXEaxR/Sjkd1wTStdpgUceUnq0nsJiY9u4uyefqDAQt4TmQMpMs1MJBGm7ZVugDjSBawaQ8xDUCWiwjlFjEuoOpr4WmxLEvVMXXfxyefzjCezrnz+DxrkJ1uEDNNvDPGczZMmB0JZuhwAww+GNok+tmpXjNgUqA1Unhg26FvzvTleNmywOmWjhoW6iJO26nYKhYqSmxLt1JwNAuBAaRZHC7Y+5UKBgxCC25IjhFy1qK7FbQiW1fD2Fhm9smSTgEikx1i7tTx1QGQdtIE37dhYBjaQ167XIyitVy9as6Q1QHlEiznuLGLVU50ZNakC3EyC0dIUJ+p1LNtUyEB6EY/qBMhZwNz7Abiji3HG3U96wsds7HFlxWmhsVPjDpcO7nEnNWbZrGQ/43hP7cN1QkmRqAWm1whfo0REsYYF6MjfYq/MpG7+TJy2O2toQLja27X+UOisa+wwDTDg/qiBQQz9xCqJRneVK5G0/EYznifVEACYeqBEwdIxYIU9rAqON9YywnupOSve5OcOFR2zAKAYRLT2E79Us2JP88Wxvs/EmVUJpKcRSttiUtCcw0y3lTiwkVtk5AYBnmjBRG0Mh7sSUnv0pAz4LXAKnVVZZnqN/2N3KxMJ12iP07Kl8se+Ghxih09N3KN1e5p09oX7W0r3FEpD+TmN9pkBb3OLrMpj8BG8mavWn8DBTIRdaPOznuZ0KJpJXyEkvSEu4oOlfT0IVf0zMs214UpqmtPW9jmHJmmxfeAN2gb2VlmjPTF3oJqYGpOWsgJaYKdpNCVXuVlmshyAb4beVBue1c6sKPNEZsT5eXR8hadEPynDRsBuNr2BZCU9El/gPaJGOCyYnakuyE4eqgVRHAY/9JP7N7101SHK4rTZKn2ravRfMt1uRSuU03ltJdrNVA8q6vRLDhfLO2RLQUE1G5DpiU/TliI2P+PbswcYmb07XpeDZYneD2BoMYA1Ere0sxb6gFbRCfiiakbcBAZFEpJLwngpq4Uh2kHigF0JWUjH9dA32FCxdfebSPxe5cOsI2kSAAIZYwN+8KPR6l1c/nP1yZVBi71cgnKzchw2bYAy0FHqfeX5bXKk/dpf7FS/WyJxAs3qf+/dhXDk78gD+DH9rZSkgKZcJ57nj75uV9dwXMg63cs/ITaP4F1GEJZRbdXAKqE8aBm+QELVzZNPBwXA4nHQt8e3N3t7EQutVWpauYvUHSIGcSLdgSgf3sKTrOyWi64Wh6yhMbVuC1c7EbODV9NoLs1GpmNjfBE+UdqEJMm+dQdM0EJGnuq3HdtkjqS6xt9XUEwxQ09O2j1X7V3Rq+DzLEnEYAxYd9PVACXylviCPyaGSHjoWENgUy33Ybmsgcsx+iIUIc5XfCx/avi+pc8z07JxHg9OMDk4aBh2WdRI+aLOi0LQf4+jBEygVy+LJnF5PCZWlLwINTo9ZLA5+AJgZcwyoNbRK29obJlxpGVw1TPMmpWZ1lQ8K+5jc1n713Tv0RGwVC7VYYrdnl/MkzEuDy1UYgD1tqpDJdq8dUjEHqbuH2WAjD34cDc3NttXTFFTTvVpJfW89Xb+SRPpKnPvE38I9dP0q5ffXr6DSxfEX60nYI9evdCv0+tVs4g+Xz9aX+OOewBVI8Wxzs7JOro6tJxj2xB/Be2DFBoX1Bb4H/I+YPnH8HgKVBDhZ+oswXfCkW1Ez+dWpFTYljodK+gOLvdevNWKbTUczsY9Zq3Q420oF1FZbbaaG1Wqab+e0jKce64wGeQEG070PtEqeniQgu/IEqgPyKBC5K0sStUrNzyBNL8oaqbs0c04JUvg3tmDqbo8/ALRhZEERrslvgTSrQ9bJ6Q9ABp6ZqMGaN5K5HS1H/1Gn/iVrAm+m5a0XUK5v/dxOWamLG9sW7W9p2AqLVpq3yoSp/SWKcMmBsGOcehpNn5WrWfM1nhlrsmUav0os6K8tPbbsxT9i6r3o9I0eH5N+AsYIormOirUwRsbjrduYgOJ51qQKKyuzZWE2Bib18uw+d1SCwAu+qQm03BwTkWNVNEtNoe/f1Tp3F1gGXgHJ+SMK+zQX5TfAT/PeU191l106aoVxun0RhTDdRfMeLWTWtro9LdgFFhRmhFXbRwsM6f9iyGR8N/W0QpnBX2wRo6ijp2Y2iVPSo7FZgwIQo9YzazBgTowDG2Qqe114GHwDJt6fb2fOn9ueKLnAitYYqWpTMZJXF8EYnWKqTr+oMAWFTAZNecF/2yArwmJ3b8SPNDIqSCabQeiOcm2UYKkU6MpAB1y+eqRMnkAWUrueLox1+/pQi6CSsrsxEJkrwiSu2sdzULYU3biOPqIdur1otCxCOkL5AlB7agi9IONUCXkFTg8pvXBci6zAm54apZhgOaPB1ujc16K0dX7phb2seZg2XQQJ8OJ08RisS7aE7V85ae5jFacmQHG5Qf3Y5Kd+ow9FA1uQ5Qt1aQI9aFmv9Z0QcdHeoC85x9Gf3aqdkmnt59ahGLFf3d6jBcZ8WIxgJ3HKWQ0XuwdND3nFzgTc6WgyM53sY12O0xtSLYQWxIVoE+Ooi3a1MjAOHYLh9S5ofzp83PTYA0LpPukNA/oReQy+lGMBlDrtRExZnXLAGLz2Qc7yG7C+nKGnpbXGv3Pm0FEBb+vw/O41ingCZhQMJSzDoggfHToRVK9XK2cRV667dQfdpe1W2J1LbqRHd09kjvduEt2TI/MvmJyiId1PO2RLzNeUEUjOKJk+Rws50lBUhPeoawCiprQQM7Hvw4e4ZPrpxCqME+YPD/ec8eu1PkTc+i2xoJLxmI5fNAg3pWW/QWmhpiX/7kzJV9pJQHwqipcyRy3IczYaDl8LdkUTECN33Z3wkMYKzMWG9g4CAPL4bROmFVgEDmHDm+JcvdEA/8xc168ySrbcDbdJOVY9MLvJZxyIfEbM9US18z7T8hkxpzMGdbi0txbKPmnSGS3SQ+8xr2LnQPKsjKv4jgdNjjjqrwJFfxu6Pu4BQFLKb8Leeh+aerv6qmIMXxvaM9YAQJtNzBIr1/ISj9B86yxGs//E7uvkfojtL8LArdvfgMi+eQbsD+qFrum+jm5GfO8FHWQoveB6jm72fAeAyt2XtfrhNFpz/curk2QZ7VlDK0pLl1kBn8F0Xak4JmVXo164CovoHhbSt2QyryGTF30GMTYJE0zeVocK6/gswtGy0MlJpyXR13eC+NaPlA9u/Xx2JvgUgpMJv60kcLxxzLc+Y17YuEnlIv8rbCfccDCx9Ma6TbP7hEc33Nev0cP0LT09tf/KSnnNEd7nIZJ8KYtEXaW183YkDAJ4UZPZ2lxHwgzXfRFcFU109RuZtH7lZTG7+60PZuDeYD33P2l3oygazzOtJ4os7ezmr6AOiutPCusfWZiE8jLTdfhozblV33s6f7RORvLWATqaoXUi7r1xv3WpGDWTM8Hu5C8ywrNK3v1BDlxREwdPT/i3txaiAJ/ob6fWN24tOxl1rn5dA6cnKoeZp5y4fkyHAsQ1sfJK1xpTYv2VM4Q6NtzpZ7rSjyhcsoaLX75+Pf36o59ki1tb3ZAYvhGxSiyctF3/dKmrv8Q3Dv7jiYKfzz79PTj5n/9oT18/Gry32AS21OkZIcGYLnCcir0K+FlokTWwMAFddLICo/sreK5Rp3uEAvTA9GSAa9vCnH6tZV5vXafzrdt+ml3aPb+L/HTXEXvcY17U0t8Ew2fdQ1ft01YxL41xepPUqGXMjhZGMfGSrrTkLtkIU7xa+ijmHAKtWT+cXpx8AvXy3xPbI/+T0dkvG3zS+bu41s9Bl+AAb0RyAYJE/S4Yn9tyZIInE18iHM0w6ahZO81IpeLTbCjNiJh2nWJUSzBiVypMp7mLQM8RcEld3PLPSfD1MrhGT1MNhS5CE8dUprLtzHjTT7GdMluYVSVyONQdSD9Q7RHsydrdHuaTU7SSa73GsHX/fD/PjD05oSeDmKEy6brXMxzpjtou/YqDEefHl5cnP+hxbi27s8GvR3+CJhRDzz36boWeeDHcAQ5XOYCAv5de+x5w5AKUR7lYwbJkSXbzqCVSela4qDZ4sXltQ1vShvas+lwhQBY3CwotS1ysY7vbuwKYK03akpO2znl4K49poQ4kkgkujr/YYrpTeztEg7mCX+KPPfG6v1rnYhVTebiuhQRQKbOkVLcgYu6HhbeNrmBzTwRzh/+HVp6ExPJ1sPwB1cUskdeKiLvRoKMQ5Ga6WGHYFy+jxxuOiLFSx4+tG9M0j/3mZhWgjVmphZzarQiXPduTxSrB8nWzvfQTNjLI2trkrXirirESWgbVivqG9elMiIKVSArEAybW/z61BlmHSd/Lean7uDyRxLfJRdo7YBhFRZaSLiXsX79nqch7A8MGdct4D4BGN2JNFYvwrausvtsIL4iurC9nP5yw61c4g+tX1oauYKX1LMM1bys8fcPp3vDbOSvd4b56+OLPsmCpW7FW4OUFck+dUWyCeQhmR4P2fYOCyfw3xp8aQE/b9vhcr8GVxr60i+pasMU9oMw+BloEdtGYdKV2DSvSGjkuoKek5lc8et9Ka0WuFm7AugNMr8F6wwvH6HMIUQ9vqvFkfaqPaDo5h/GmuOrCp1GiBmCkoMZGFyuCnkcJxp4ZCVu7+3zRkf3K3sOBO9o9t8B9mfocgn9c3GxQ/TmncsfVa/lhFAWhrOCgE3QNaLO9xSqLcQGmNmwLu87pm+1qPBhIW8/2qseckx20uz7eO0vw/0B9utC5aeBJX6x+j0NPa5FW2LRU66yvUF/TTTrAI5kvHeiaBtkgUZ2IRNYDokmNWZbvhiVZguw7Rm1eNn+3uyElX75wwJRf+cK6derlC+vLjMxubeR3TDahP9io1KxinSF+GE1kVS4sMZXURIbpnNcuQrvTXlI0iA5JwhP99K7x3g+Z+It6aMsb4ukuCq9l4HdDMu2xomQp1DdW6l1i1fRvNQTm6p7+pWBfxSbHWH8Z33hLUG+4+8LUImHA/p0/zrOwiE4VKENSKPJERBR+oSXSzryUkidSShWKCn184ismvvjjyKfL0x9Pv1552rhf0OLq5OJLT5PakSV4HSxNQAenggBWNAhwcEFgTwT/u05feeKTQAHqtqX+YaCz5RLjPKT0iruNgJWvebXKIvHFHTxLmNHCSyFufeXVJSxapIsXv/Vdml0fgcl6vubC1zlKwhd/v0WW0rToDPx1aj5WT26nrPSlwYtnqhz8IFKAB9w/geoTgH17/crrfAlG+pr2rOtXTSr/9SvhHJbWEff1r6L8jVlH9c0rYZw4y7S++Aa9MKnTPpLnHBMUmCN5UbzmRh1P9664E12xEsTbbolXfcvbdrHfTcF1nyeurLxHoBaF9fi3zE7GRqPDcW+t1vl09nb/LX4vZkdddQ0O1j58275nALcmSGHM/2CdW0XUjSLqPhF1mYhn+lQKfclF/5DLN7/D0hpw+3s0MsvlaTQe7Q9HHv0Ziz/7nqjcHMnXLsgjzbb/mL/ukWsuQeJ+3zVIHU+BSMpiTEMvVTek6mByX5hj8iIMdztHzDXkgnU6e+rPFNJOg3dP7eyGi0Jj+wA4YIsSFobeyBt7+96Bd9iFq38rI0QNNsAIQE3If7VQ1VV38eixBXmvJ4EbHanL2x6s/XETUfDrG7h3pWBpkSPu/7mrQIlCVoK0mms+J8brUE03V9aXojoxY46zwKPB0Iu8VCir/sb2xy66mfH9sC7+AKVqIhgTeHK2T8q7E2HRyBPfGvTX40OMxLu7kPDcvamUHKMd5iOoh1w/v4Nt2SjknhhWa2G/fQVK7Z/lfsdDm7fdst17AWO6onx/bFxaQJW2HneslWO4MOSK3U3bSQVsG3WmNoKxtY+zN28bLgY15J2J/fWGop6xRosHztTpGyKNu2mHJdavewYMHLivG8GXaS8/Db0tDEj6IKelyH9GfLnvqUC7s3QIHehXNqiGJEfld50My01gYL274o6caAHdJ4ZuNZ3odlFWlkTMeBWL6c4T850tHXlisG/xKuihtz8y8YhYVjgYe+PDQxOD+G+v9ahvDPFi15ThFPnCAel0DvEbMqWcBtZuidt3WKL8byaCySWxu/OaiWYusO2L3jypcF6il3sVRxFPg2y5BMVE5Dh9YIfaDlzG5DD5JgkAFXXJUbiU4iSuHmtCFJ+xkmqvf8WRLmGCPyhXm0PhvGqdax3KC6eg0N2zqzovXPO5+doXxoSv6oGNdK3i2zXae85Qe+xubTgZ2FaxrnqSkWcF+peKAtgghiCfSUe3Oyq6bdbQ97STtrbbubtVXmXFfdMHelAnQdlwJ77wUt8Ga75QSr8ddpfJ1QgXOpq2TCmBG7U9oboiKHktbOdUNtZvfcCHMLu1exo7ouvnPb68tD3Ra9fD1rjpspbBJ6568DGkX5t16pqRrq/31fP/AWhzbDRZdgAA"
E1_EXPECTED_FILES = {"p1_e1.py", "p1_e1_tests.py"}

PROJECT_DIR = Path(PROJECT_DIR).expanduser().resolve()
archive = gzip.decompress(base64.b64decode(_E1_EMBEDDED))
if hashlib.sha256(archive).hexdigest() != E1_BUNDLE_SHA256:
    raise RuntimeError("E1 embedded bundle hash mismatch")
embedded = json.loads(archive)
if set(embedded) != E1_EXPECTED_FILES:
    raise RuntimeError("Unexpected E1 code files")
E1_CODE_DIR = PROJECT_DIR / "experiment_code" / ("e1-"+E1_BUNDLE_SHA256[:16])
E1_CODE_DIR.mkdir(parents=True, exist_ok=True)
for name, content in embedded.items():
    dst = E1_CODE_DIR / name
    if dst.exists() and dst.read_text() != content:
        raise RuntimeError("Existing E1 code differs. Preserve it; do not overwrite: " + str(dst))
    if not dst.exists():
        temp = dst.with_name(name+".tmp-"+str(os.getpid()))
        temp.write_text(content)
        os.replace(temp, dst)
print("E1 code:", E1_CODE_DIR)
print("Bundle hash:", E1_BUNDLE_SHA256)


## Check cores, available RAM, Qoala revision, interpreter, and offline tests

In [ ]:
import os, sys, shutil, subprocess

assert MODE in ("pilot", "full")
assert isinstance(WORKERS, int) and not isinstance(WORKERS, bool) and WORKERS >= 1
available_cpus = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
if available_cpus and WORKERS > available_cpus:
    raise RuntimeError(f"WORKERS={WORKERS}, but Jupyter has only {available_cpus} available CPU cores")

RUN_ENV = os.environ.copy()
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                 "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    RUN_ENV[variable] = "1"
RUN_ENV["PYTHONUNBUFFERED"] = "1"
RUN_ENV.pop("PYTHONOPTIMIZE", None)

# Defend against another memory failure. The reported memory is the approximate
# available budget, NOT the installed physical RAM amount.
def estimated_available_mib():
    candidates=[]
    meminfo=Path("/proc/meminfo")
    if meminfo.exists():
        for line in meminfo.read_text().splitlines():
            if line.startswith("MemAvailable:"):
                candidates.append(int(line.split()[1]) / 1024)
                break
    cg_max=Path("/sys/fs/cgroup/memory.max")
    cg_used=Path("/sys/fs/cgroup/memory.current")
    if cg_max.exists() and cg_used.exists() and cg_max.read_text().strip() != "max":
        candidates.append(max(0,(int(cg_max.read_text())-int(cg_used.read_text()))/1048576))
    return min(candidates) if candidates else None

free_mib=estimated_available_mib()
required_mib=WORKERS*1600+2048
print("CPU cores visible:", available_cpus)
print("Estimated free RAM:", "unknown" if free_mib is None else f"{free_mib/1024:.1f} GiB")
print("E1 safety budget:", f"{required_mib/1024:.1f} GiB available for {WORKERS} workers")
if free_mib is not None and free_mib < required_mib:
    raise RuntimeError("Insufficient available RAM for requested workers. Reduce WORKERS "+
                       "or run when P1_Resume_After_MemoryError is not consuming memory.")

PARENT_SOURCE_HASH = "470078ab4b2c014d42b97dbba496934a0195ae13a58d3404274c06b8e5b3b7b4"
Q12_CODE_DIR = Path(Q12_CODE_DIR).expanduser().resolve() if Q12_CODE_DIR else PROJECT_DIR / "experiment_code" / ("q12-" + PARENT_SOURCE_HASH[:16])
Q12_RUN_DIR = Path(Q12_RUN_DIR).expanduser().resolve()
if not (Q12_CODE_DIR / "q12.py").is_file():
    raise RuntimeError("Original Q12 code not found: " + str(Q12_CODE_DIR))
if not (Q12_RUN_DIR / "COMPLETE.json").is_file():
    raise RuntimeError("Original complete Q12 run not found: " + str(Q12_RUN_DIR))
RUN_ENV["P1_Q12_CODE_DIR"] = str(Q12_CODE_DIR)

repo_candidates=([Path(REPO).expanduser()] if REPO else
                 [PROJECT_DIR/name for name in ("qoala-sim","p1-qoala","qoala")] +
                 [Path.home()/"p1-qoala",Path.home()/"qoala-sim"])
repos=[]
for candidate in repo_candidates:
    if not candidate.is_dir(): continue
    r=subprocess.run(["git","-C",str(candidate),"rev-parse","HEAD"],capture_output=True,text=True)
    if r.returncode==0 and r.stdout.strip()=="31fa2c9699578d227736557f35cd01a134c4bbde":
        repos.append(candidate.resolve())
if not repos: raise RuntimeError("Pinned Qoala checkout not found; set REPO in first code cell")
REPO=repos[0]

if PYTHON:
    python_candidates=[str(Path(PYTHON).expanduser())]
else:
    python_candidates=[str(PROJECT_DIR/".venv-p1"/"bin"/"python"),
                       str(PROJECT_DIR/".venv"/"bin"/"python"),sys.executable]
python_candidates=list(dict.fromkeys(python_candidates))
preflight="import json,sys; import q6_reference; print(json.dumps(q6_reference.check_repo(sys.argv[1])))"
errors=[];chosen=None
for candidate in python_candidates:
    if not Path(candidate).is_file():continue
    r=subprocess.run([candidate,"-c",preflight,str(REPO)],cwd=str(Q12_CODE_DIR),env=RUN_ENV,capture_output=True,text=True)
    if r.returncode==0:
        chosen=candidate
        print("Validated interpreter:",chosen)
        print("Pinned Qoala checkout:",REPO)
        print(r.stdout)
        break
    errors.append(candidate+"\n"+r.stderr[-1200:])
if chosen is None:
    print("\n".join(errors))
    raise RuntimeError("No valid Q12 NetSquid/Qoala interpreter found; specify PYTHON")
PYTHON=chosen
for script, cwd in ((Q12_CODE_DIR/"q12_tests.py",Q12_CODE_DIR),
                    (E1_CODE_DIR/"p1_e1_tests.py",E1_CODE_DIR)):
    r=subprocess.run([PYTHON,"-u",str(script)],cwd=str(cwd),env=RUN_ENV,capture_output=True,text=True)
    print(r.stdout)
    if r.returncode:
        print(r.stderr)
        raise RuntimeError("Offline checks failed: "+str(script))
print("Preflight passed. No new NetSquid simulation has run yet.")


## Start or resume: pilot first, then full with the same E1 run directory

In [ ]:
import json, os, signal, subprocess, time
from datetime import datetime, timezone

E1_RESULTS = PROJECT_DIR / "results"
E1_RESULTS.mkdir(parents=True,exist_ok=True)
POINTER = E1_RESULTS / (".e1-latest-"+E1_BUNDLE_SHA256[:16]+".json")

def choose_run_dir(choice):
    if choice=="auto" and POINTER.exists():
        info=json.loads(POINTER.read_text())
        assert info["bundle_sha256"]==E1_BUNDLE_SHA256
        r=Path(info["run_dir"]).expanduser().resolve()
        if not r.is_dir():raise RuntimeError("Saved E1 run folder vanished: "+str(r))
        return r
    if choice in (None,"auto"):
        stamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
        return E1_RESULTS/("e1-"+stamp)
    r=Path(choice).expanduser().resolve()
    if not r.is_dir() or not r.name.startswith("e1-"):
        raise RuntimeError("Explicit E1 resume directory does not exist: "+str(r))
    return r

RUN_DIR = choose_run_dir(RESUME_RUN_DIR)
if any(RUN_DIR==d or d in RUN_DIR.parents for d in [Q12_RUN_DIR, REPO, Q12_CODE_DIR, E1_CODE_DIR]):
    raise RuntimeError("E1 results cannot be stored inside parent code/results directories")
RUN_DIR.mkdir(parents=True,exist_ok=True)
tmp=POINTER.with_name(POINTER.name+".tmp-"+str(os.getpid()))
tmp.write_text(json.dumps(dict(run_dir=str(RUN_DIR),bundle_sha256=E1_BUNDLE_SHA256),indent=2))
os.replace(tmp,POINTER)
print("E1 RUN DIRECTORY:",RUN_DIR)
print("Mode:",MODE,"| Workers:",WORKERS)
print("To resume exactly, retain:",RUN_DIR)
launcher_log=RUN_DIR/("launcher-"+datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")+".log")
cmd=[PYTHON,"-u",str(E1_CODE_DIR/"p1_e1.py"),"run","--project",str(PROJECT_DIR),
     "--q12-run",str(Q12_RUN_DIR),"--q12-code",str(Q12_CODE_DIR),
     "--repo",str(REPO),"--run-dir",str(RUN_DIR),"--mode",MODE,"--workers",str(WORKERS)]
with launcher_log.open("w") as log:
    process=subprocess.Popen(cmd,cwd=str(E1_CODE_DIR),env=RUN_ENV,
                             stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                             text=True,bufsize=1,start_new_session=True)
    try:
        for line in process.stdout:
            log.write(line);log.flush();print(line,end="",flush=True)
        code=process.wait()
    except BaseException:
        for sig in (signal.SIGINT,signal.SIGTERM,signal.SIGKILL):
            if process.poll() is not None:break
            try:os.killpg(process.pid,sig)
            except ProcessLookupError:break
            try:process.wait(timeout=8);break
            except subprocess.TimeoutExpired:pass
        raise
    finally:process.stdout.close()
if code:
    raise RuntimeError(f"E1 stopped with exit code {code}. Check {launcher_log}; completed batches can be resumed")
print("E1 stage finished. Run the review-export cell.")


## Review and compact export

Run this cell after a successful pilot or full run. If the simulation aborts, completed batches remain on the server; restart the notebook with `RESUME_RUN_DIR="auto"` and the same mode.

In [ ]:
import zipfile
from IPython.display import FileLink, display

if "RUN_DIR" not in globals():
    raise RuntimeError("Run the configuration, preflight, and run cells first")
print("Run:",RUN_DIR)
for file in ("PILOT_PASSED.json","COMPLETE.json","analysis.json"):
    p=RUN_DIR/file
    if p.is_file():
        print(file,":",p.stat().st_size,"bytes")
        if file=="analysis.json":
            data=json.loads(p.read_text())
            for x in data["contrasts"]:
                print(x["contrast"],"delta(pp)=",round(x["campaign_mean_difference_pp"],4),
                      "CI=",[round(v,4) for v in x["interval_pp"]],
                      "primary=",x["primary"])

zip_path=RUN_DIR/("E1_Review_"+("complete" if (RUN_DIR/"COMPLETE.json").is_file() else "pilot")+".zip")
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:
    include=[p for p in RUN_DIR.rglob("*") if p.is_file() and (p.name.endswith(".json") or
              p.name.endswith(".log") or (EXPORT_COMPACT and p.name.endswith(".compact.jsonl.gz")))]
    for p in sorted(include):
        if ".tmp-" in p.name: continue
        z.write(p,p.relative_to(RUN_DIR))
    for p in (E1_CODE_DIR/"p1_e1.py",E1_CODE_DIR/"p1_e1_tests.py"):
        z.write(p,"source/"+p.name)
print("Review archive:",zip_path)
print("Raw simulator event records are preserved in",RUN_DIR,"but excluded from this compact archive")
display(FileLink(str(zip_path)))
